In [2]:
# ===============================================
# Airbnb Boston Dataset Joining Pipeline
# ===============================================

import pandas as pd

# ===============================================
# Load Cleaned Datasets
# ===============================================

print("=" * 60)
print("Loading cleaned datasets...")
print("=" * 60)

listings = pd.read_csv(
    "D:/Talent assessment/Data/cleaned/listings_clean.csv"
)

calendar = pd.read_csv(
    "D:/Talent assessment/Data/cleaned/calendar_clean.csv"
)

reviews = pd.read_csv(
    "D:/Talent assessment/Data/cleaned/reviews_clean.csv"
)

neighbourhoods = pd.read_csv(
    "D:/Talent assessment/Data/cleaned/neighbourhoods_clean.csv"
)

print("\nDatasets loaded successfully!")

print("\nDataset Shapes")
print("-" * 40)
print("Listings       :", listings.shape)
print("Calendar       :", calendar.shape)
print("Reviews        :", reviews.shape)
print("Neighbourhoods :", neighbourhoods.shape)


# ===============================================
# Aggregate Calendar
# ===============================================

print("\nAggregating calendar data...")

calendar["date"] = pd.to_datetime(
    calendar["date"],
    errors="coerce"
)

# Convert availability if needed
if calendar["available"].dtype == object:
    calendar["available"] = calendar["available"].map({
        "t": 1,
        "f": 0,
        True: 1,
        False: 0
    })

calendar["available"] = calendar["available"].fillna(0)

calendar_summary = (
    calendar
    .groupby("listing_id")
    .agg(
        total_calendar_days=("date", "count"),
        available_days=("available", "sum"),
        avg_minimum_nights=("minimum_nights", "mean"),
        avg_maximum_nights=("maximum_nights", "mean")
    )
    .reset_index()
)

calendar_summary["occupancy_rate"] = (
    1 -
    (
        calendar_summary["available_days"] /
        calendar_summary["total_calendar_days"]
    )
)

calendar_summary["availability_rate"] = (
    calendar_summary["available_days"] /
    calendar_summary["total_calendar_days"]
)

print("Calendar Summary Shape:", calendar_summary.shape)


# ===============================================
# Aggregate Reviews
# ===============================================

print("\nAggregating reviews data...")

reviews["date"] = pd.to_datetime(
    reviews["date"],
    errors="coerce"
)

review_summary = (
    reviews
    .groupby("listing_id")
    .agg(
        total_reviews=("id", "count"),
        first_review=("date", "min"),
        latest_review=("date", "max")
    )
    .reset_index()
)

print("Review Summary Shape:", review_summary.shape)


# ===============================================
# Join Listings + Calendar Summary
# ===============================================

print("\nJoining Listings + Calendar Summary...")

master = listings.merge(
    calendar_summary,
    left_on="id",
    right_on="listing_id",
    how="left"
)

print(master.shape)


# ===============================================
# Join Review Summary
# ===============================================

print("\nJoining Review Summary...")

master = master.merge(
    review_summary,
    left_on="id",
    right_on="listing_id",
    how="left",
    suffixes=("", "_review")
)

print(master.shape)


# ===============================================
# Join Neighbourhood Dataset
# ===============================================

print("\nJoining Neighbourhood dataset...")

print("Neighbourhood Columns:")
print(neighbourhoods.columns.tolist())

if "neighbourhood" in neighbourhoods.columns:

    master = master.merge(
        neighbourhoods,
        left_on="neighbourhood_cleansed",
        right_on="neighbourhood",
        how="left"
    )

print(master.shape)


# ===============================================
# Remove Duplicate Columns
# ===============================================

master = master.loc[:, ~master.columns.duplicated()]


# ===============================================
# Missing Values
# ===============================================

print("\nTop 20 Missing Values")
print("-" * 40)

print(
    master.isnull()
    .sum()
    .sort_values(ascending=False)
    .head(20)
)


# ===============================================
# Dataset Summary
# ===============================================

print("\nMaster Dataset Summary")
print("-" * 40)

print(master.info())

print("\nMaster Dataset Shape")
print(master.shape)

print("\nFirst Five Records")
print(master.head())


# ===============================================
# Save Master Dataset
# ===============================================

master.to_csv(
    "D:/Talent assessment/Data/cleaned/master_dataset.csv",
    index=False
)

print("\n" + "=" * 60)
print("Dataset joining completed successfully!")
print("Saved as master_dataset.csv")
print("=" * 60)

Loading cleaned datasets...

Datasets loaded successfully!

Dataset Shapes
----------------------------------------
Listings       : (3693, 78)
Calendar       : (1611110, 5)
Reviews        : (252910, 6)
Neighbourhoods : (26, 2)

Aggregating calendar data...
Calendar Summary Shape: (4414, 7)

Aggregating reviews data...
Review Summary Shape: (3500, 4)

Joining Listings + Calendar Summary...
(3693, 85)

Joining Review Summary...
(3693, 89)

Joining Neighbourhood dataset...
Neighbourhood Columns:
['neighbourhood_group', 'neighbourhood']
(3693, 91)

Top 20 Missing Values
----------------------------------------
neighbourhood_group_cleansed    3693
neighbourhood_group             3693
host_about                      1388
host_location                    928
license                          793
first_review_review              616
total_reviews                    616
first_review                     616
review_scores_rating             616
listing_id_review                616
latest_review  

In [2]:
# ===============================================
# Aggregate Reviews
# ===============================================

print("Aggregating Reviews...")


reviews_summary = (
    reviews
    .groupby("listing_id")
    .agg(
        review_count=("id", "count"),
        last_review_date=("date", "max")
    )
    .reset_index()
)


print(reviews_summary.shape)

reviews_summary.head()

Aggregating Reviews...
(9432, 3)


,listing_id,review_count,last_review_date
0,28871,799,2026-06-01
1,29051,906,2026-06-01
2,44129,186,2026-04-30
3,44391,42,2022-08-20
4,48373,5,2024-04-28
